# Asimov's Mind — Multi-Condition Experiment Analysis

Comparing three experimental conditions:
- **Condition A**: Ungoverned single agent (autoresearch baseline)
- **Condition B**: Governed single agent (Three Laws constraints)
- **Condition C**: Governed multi-agent swarm (3 specialists + Three Laws)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

results_dir = Path("governed/results")

# Load main results (from experiment_runner.py v2)
results_path = results_dir / "all_results.tsv"
if results_path.exists():
    df = pd.read_csv(results_path, sep="\t")
    df["val_bpb"] = pd.to_numeric(df["val_bpb"], errors="coerce")
    df["peak_vram_mb"] = pd.to_numeric(df["peak_vram_mb"], errors="coerce")
    print(f"Loaded {len(df)} result rows from all_results.tsv")
    print(f"Phases: {df['phase'].value_counts().to_dict()}")
    print(f"Conditions: {df['condition'].value_counts().to_dict()}")
else:
    print(f"No results file at {results_path}")
    print("Run: python governed/experiment_runner.py all")
    df = pd.DataFrame()

# Load reconstructed history (from git branches)
histories = {}
for cond in ["A", "B", "C"]:
    hist_path = results_dir / f"history_{cond}.tsv"
    if hist_path.exists():
        histories[cond] = pd.read_csv(hist_path, sep="\t")
        histories[cond]["val_bpb"] = pd.to_numeric(histories[cond]["val_bpb"], errors="coerce")
        print(f"History {cond}: {len(histories[cond])} experiments")

df.head(10)

In [ ]:
# --- Reconstructed History: Crash rates per condition (from prior AI agent runs) ---
if histories:
    print("=" * 70)
    print("RECONSTRUCTED HISTORY (from git branches + prior TSV data)")
    print("=" * 70)
    for cond, hist in sorted(histories.items()):
        counts = hist["status"].value_counts()
        total = len(hist)
        crashes = counts.get("crash", 0)
        keeps = counts.get("keep", 0)
        discards = counts.get("discard", 0)
        print(f"\nCondition {cond}:  {total} experiments")
        print(f"  Keep: {keeps}  Discard: {discards}  Crash: {crashes}  "
              f"Crash rate: {crashes/total:.0%}")
    print()

# --- New experiment outcomes ---
if not df.empty:
    print("=" * 70)
    print("NEW EXPERIMENT RESULTS (from experiment_runner.py v2)")
    print("=" * 70)

    baseline = df[df["phase"] == "baseline"]
    if not baseline.empty:
        bl = baseline.iloc[0]
        print(f"\nBaseline val_bpb: {bl['val_bpb']:.6f}  VRAM: {bl['peak_vram_mb']:.0f}MB")

    isolated = df[df["phase"] == "isolated"]
    if not isolated.empty:
        print(f"\nIsolated experiments: {len(isolated)} rows")
        for cond in ["A", "B", "C"]:
            cond_data = isolated[isolated["condition"] == cond]
            counts = cond_data["status"].value_counts()
            print(f"  Condition {cond}: {counts.to_dict()}")

In [ ]:
# --- Isolated experiment results: which changes helped? ---
if not df.empty:
    isolated = df[(df["phase"] == "isolated") & (df["condition"] == "A")]  # same for all conditions
    baseline_row = df[df["phase"] == "baseline"]
    if not baseline_row.empty and not isolated.empty:
        bl_bpb = baseline_row.iloc[0]["val_bpb"]
        valid = isolated[isolated["val_bpb"] > 0].copy()
        valid["delta"] = bl_bpb - valid["val_bpb"]
        valid = valid.sort_values("delta", ascending=False)

        print(f"{'Rank':>4}  {'Exp':>3}  {'val_bpb':>10}  {'Delta':>9}  {'VRAM MB':>8}  Description")
        print("-" * 75)
        for rank, (_, row) in enumerate(valid.iterrows(), 1):
            marker = ">>>" if row["delta"] > 0 else "   "
            print(f"{rank:4d}  {row['experiment']:3.0f}  {row['val_bpb']:10.6f}  "
                  f"{row['delta']:+9.6f}  {row['peak_vram_mb']:8.0f}  {marker} {row['description']}")

        improved = valid[valid["delta"] > 0]
        print(f"\n{len(improved)}/{len(valid)} experiments improved over baseline {bl_bpb:.6f}")

## Isolated Experiments: Parameter Sensitivity

Bar chart showing each parameter change's effect on val_bpb relative to baseline.

In [ ]:
if not df.empty:
    isolated = df[(df["phase"] == "isolated") & (df["condition"] == "A")]
    baseline_row = df[df["phase"] == "baseline"]

    if not baseline_row.empty and not isolated.empty:
        bl_bpb = baseline_row.iloc[0]["val_bpb"]
        valid = isolated[isolated["val_bpb"] > 0].copy()
        valid["delta"] = bl_bpb - valid["val_bpb"]
        valid = valid.sort_values("delta", ascending=False)

        fig, ax = plt.subplots(figsize=(14, 6))
        colors = ["#2ecc71" if d > 0 else "#e74c3c" for d in valid["delta"]]
        bars = ax.bar(range(len(valid)), valid["delta"], color=colors, edgecolor="black", linewidth=0.5)

        ax.set_xticks(range(len(valid)))
        ax.set_xticklabels(valid["description"], rotation=45, ha="right", fontsize=9)
        ax.set_ylabel("Improvement (baseline - val_bpb)", fontsize=11)
        ax.set_title(f"Parameter Sensitivity: Change in val_bpb vs Baseline ({bl_bpb:.6f})", fontsize=13)
        ax.axhline(y=0, color="black", linewidth=0.8)
        ax.grid(axis="y", alpha=0.3)

        for bar, delta in zip(bars, valid["delta"]):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f"{delta:+.4f}", ha="center", va="bottom" if delta > 0 else "top",
                    fontsize=8, fontweight="bold")

        plt.tight_layout()
        plt.savefig("param_sensitivity.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("Saved to param_sensitivity.png")

## Cumulative Results: Condition Comparison

How does each condition's cumulative parameter stacking compare?

In [ ]:
if not df.empty:
    cumulative = df[df["phase"].isin(["cumulative", "optimal"])]
    baseline_row = df[df["phase"] == "baseline"]

    if not baseline_row.empty and not cumulative.empty:
        bl_bpb = baseline_row.iloc[0]["val_bpb"]

        fig, ax = plt.subplots(figsize=(14, 7))
        cond_colors = {"A": "#e74c3c", "B": "#3498db", "C": "#2ecc71", "optimal": "#9b59b6"}
        cond_labels = {
            "A": "Ungoverned (replay)",
            "B": "Governed Single (replay)",
            "C": "Governed Swarm (replay)",
            "optimal": "Optimal Order",
        }

        for cond in ["A", "B", "C", "optimal"]:
            cond_data = cumulative[(cumulative["condition"] == cond) & (cumulative["val_bpb"] > 0)]
            if cond_data.empty:
                continue
            steps = list(range(len(cond_data) + 1))
            bpbs = [bl_bpb] + list(cond_data["val_bpb"])
            ax.plot(steps, bpbs, marker="o", color=cond_colors[cond],
                    linewidth=2, markersize=6, label=f"{cond_labels[cond]}")
            # Annotate final value
            ax.annotate(f"{bpbs[-1]:.4f}", (steps[-1], bpbs[-1]),
                       textcoords="offset points", xytext=(8, 0), fontsize=9,
                       color=cond_colors[cond], fontweight="bold")

        ax.axhline(y=bl_bpb, color="gray", linestyle="--", alpha=0.5, label=f"Baseline ({bl_bpb:.4f})")
        ax.set_xlabel("Cumulative Step", fontsize=11)
        ax.set_ylabel("val_bpb (lower is better)", fontsize=11)
        ax.set_title("Cumulative Progress: Historical Replay vs Optimal Order", fontsize=13)
        ax.legend(fontsize=10)
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig("cumulative_comparison.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("Saved to cumulative_comparison.png")

## Historical Crash Rate Comparison

The paper's central metric: does governance reduce wasted experiments?

In [ ]:
if histories:
    fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
    cond_colors = {"A": "#e74c3c", "B": "#3498db", "C": "#2ecc71"}
    cond_labels = {"A": "A: Ungoverned", "B": "B: Governed Single", "C": "C: Governed Swarm"}

    for ax, (cond, hist) in zip(axes, sorted(histories.items())):
        counts = hist["status"].value_counts()
        categories = ["keep", "discard", "crash"]
        values = [counts.get(c, 0) for c in categories]
        cat_colors = ["#2ecc71", "#f39c12", "#e74c3c"]

        bars = ax.bar(categories, values, color=cat_colors, edgecolor="black", linewidth=0.5)
        for bar, v in zip(bars, values):
            if v > 0:
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                        str(v), ha="center", fontweight="bold", fontsize=12)

        total = len(hist)
        crash_pct = counts.get("crash", 0) / total * 100
        ax.set_title(f"{cond_labels[cond]}\n({total} exps, {crash_pct:.0f}% crash rate)", fontsize=11)
        ax.set_ylabel("Count" if cond == "A" else "")
        ax.set_ylim(0, max(values) + 1.5)

    plt.suptitle("Experiment Outcomes by Condition (Reconstructed from Git History)", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig("crash_rate_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved to crash_rate_comparison.png")

In [ ]:
# --- Final summary table for the paper ---
print("=" * 70)
print("SUMMARY TABLE (for paper)")
print("=" * 70)

if not df.empty:
    baseline_row = df[df["phase"] == "baseline"]
    bl_bpb = baseline_row.iloc[0]["val_bpb"] if not baseline_row.empty else 0

    print(f"\nBaseline val_bpb: {bl_bpb:.6f}")
    print()

    # Cumulative final results (including optimal)
    cumulative = df[df["phase"].isin(["cumulative", "optimal"])]
    print(f"{'Condition':<30} {'Best val_bpb':>12} {'Improvement':>12} {'Steps':>6}")
    print("-" * 65)
    for cond, label in [("A", "Ungoverned (replay)"),
                        ("B", "Governed Single (replay)"),
                        ("C", "Governed Swarm (replay)"),
                        ("optimal", "Optimal Order")]:
        cond_data = cumulative[(cumulative["condition"] == cond) & (cumulative["val_bpb"] > 0)]
        if cond_data.empty:
            print(f"{cond}: {label:<25} {'N/A':>12} {'N/A':>12} {'N/A':>6}")
        else:
            best = cond_data["val_bpb"].min()
            delta = bl_bpb - best
            n_steps = len(cond_data)
            print(f"{cond}: {label:<25} {best:>12.6f} {delta:>+12.6f} {n_steps:>6}")

if histories:
    print("\nHistorical crash rates (from AI agent runs):")
    for cond in ["A", "B", "C"]:
        if cond in histories:
            h = histories[cond]
            crashes = (h["status"] == "crash").sum()
            print(f"  Condition {cond}: {crashes}/{len(h)} = {crashes/len(h):.0%} crash rate")